# 04 – StruQ (Alpaca-cleaned) preprocessing
**Dataset:** the instruction-tuning data and attack templates of StruQ (Chen et al., USENIX Security 2025, ref. [8])
**Source:** code https://github.com/Sizhe-Chen/StruQ (licence file: **CC BY-NC 4.0, non-commercial**); data `alpaca_data_cleaned.json` from https://github.com/gururise/AlpacaDataCleaned (the file StruQ's `run.py` trains on), derived from Stanford Alpaca.
**Project:** Cyber Nexus

StruQ builds on Alpaca: samples that have a non-empty `input` field are treated as "instruction + data", and attacks inject extra instructions into the data. We process:

* **Part A** – the Alpaca-cleaned samples that have an `input` (these play the role of benign documents, including natural instruction-like text).
* **Part B** – the attack templates defined in StruQ's `config.py`.

Not available here: AlpacaEval's `davinci_003_outputs.json` (Hugging Face host, unreachable); it is the utility-evaluation file and is not needed for our attack/false-positive comparison.

In [1]:
import json, ast, re, unicodedata, hashlib, subprocess, html
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
md5 = lambda s: hashlib.md5(s.encode("utf-8")).hexdigest()

ZW = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")
CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def clean_text(t, collapse_spaces=False):
    t = unicodedata.normalize("NFC", str(t))
    t = ZW.sub("", t); t = CTRL.sub("", t)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = "\n".join(line.rstrip() for line in t.split("\n"))
    t = re.sub(r"\n{3,}", "\n\n", t)
    if collapse_spaces: t = re.sub(r"[ \t]{2,}", " ", t)
    return t.strip()

def deva_ratio(t):
    letters = [c for c in t if c.isalpha()]
    return sum("\u0900" <= c <= "\u097f" for c in letters) / max(len(letters), 1)

def lang_tag(t):
    r = deva_ratio(t)
    return "hi" if r > 0.5 else ("mixed" if r > 0.02 else "en")   # Latin-script Hinglish needs a separate detector

def group_split(df, group_col, strat_col=None, seed=SEED):
    """70/15/15 split where all rows of one group stay together (avoids leakage)."""
    cols = [group_col] + ([strat_col] if strat_col else [])
    g = df.drop_duplicates(group_col)[cols]
    s = g[strat_col] if strat_col else None
    g_tr, g_tmp = train_test_split(g, test_size=0.30, stratify=s, random_state=seed)
    s2 = g_tmp[strat_col] if strat_col else None
    g_va, g_te = train_test_split(g_tmp, test_size=0.50, stratify=s2, random_state=seed)
    parts = [df[df[group_col].isin(x[group_col])].copy().assign(split=n) for x, n in [(g_tr, "train"), (g_va, "val"), (g_te, "test")]]
    for a in range(3):
        for b in range(a + 1, 3):
            assert not (set(parts[a][group_col]) & set(parts[b][group_col])), "group leakage!"
    assert sum(len(p) for p in parts) == len(df)
    return parts

def add_buckets(parts, len_col, by=None):
    """short/medium/long using tertiles computed on the TRAIN part only."""
    tr = parts[0]
    if by:
        q = tr.groupby(by)[len_col].quantile([1/3, 2/3]).unstack(); q.columns = ["q1", "q2"]
        f = lambda r: ("short" if r[len_col] <= q.loc[r[by], "q1"] else "medium" if r[len_col] <= q.loc[r[by], "q2"] else "long")
    else:
        q1, q2 = tr[len_col].quantile([1/3, 2/3]); q = pd.Series({"q1": q1, "q2": q2})
        f = lambda r: "short" if r[len_col] <= q1 else "medium" if r[len_col] <= q2 else "long"
    for p in parts: p["length_bucket"] = p.apply(f, axis=1)
    return q

def git_commit(repo):
    return subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

def ensure_clone(url, dest):
    if not Path(dest).exists():
        subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True)

REPO = Path("StruQ"); DATA = REPO / "data"; DATA.mkdir(exist_ok=True)
OUT = Path("processed/04_struq_alpaca"); OUT.mkdir(parents=True, exist_ok=True)
ensure_clone("https://github.com/Sizhe-Chen/StruQ.git", REPO)
commit = git_commit(REPO); print("StruQ commit:", commit)
ALPACA_URL = "https://raw.githubusercontent.com/gururise/AlpacaDataCleaned/refs/heads/main/alpaca_data_cleaned.json"
ALPACA = DATA / "alpaca_data_cleaned.json"
if not ALPACA.exists():
    import urllib.request; urllib.request.urlretrieve(ALPACA_URL, ALPACA)
print("alpaca file:", round(ALPACA.stat().st_size / 1e6, 1), "MB")

StruQ commit: e4372b9e3d45ecdfc7c0ad70662094efe6e3850b
alpaca file: 44.3 MB


## Part A – Alpaca-cleaned
### A.1 Inspect

In [2]:
raw = pd.DataFrame(json.load(open(ALPACA, encoding="utf-8")))
print("RAW shape:", raw.shape, "| columns:", raw.columns.tolist()); print(raw.dtypes.value_counts().to_dict())
print("missing:", raw.isna().sum().to_dict())
has_in = raw.input.str.strip() != ""
print(f"rows with an input field: {int(has_in.sum())} ({has_in.mean():.1%}); without: {int((~has_in).sum())}")
print("empty instruction:", int((raw.instruction.str.strip() == "").sum()), "| empty output:", int((raw.output.str.strip() == "").sum()))
print("duplicate (instruction, input):", int(raw.duplicated(["instruction", "input"]).sum()))
r = raw[has_in]
print("input length (words):", r.input.str.split().str.len().describe().round(1).to_dict())
print("non-ASCII rows (input):", int(r.input.map(lambda s: any(ord(c) > 127 for c in s)).sum()), "| zero-width/control chars:", int(r.input.str.contains(r"[\u200b\u200c\u200d\u2060\ufeff\x00-\x08\x0b\x0c\x0e-\x1f]").sum()))
print("distinct input texts among rows with input:", r.input.nunique(), "of", len(r))

RAW shape: (51760, 3) | columns: ['instruction', 'input', 'output']
{<StringDtype(storage='python', na_value=nan)>: 3}
missing: {'instruction': 0, 'input': 0, 'output': 0}
rows with an input field: 19157 (37.0%); without: 32603


empty instruction: 0 | empty output: 0
duplicate (instruction, input): 13
input length (words): {'count': 19157.0, 'mean': 11.3, 'std': 17.0, 'min': 1.0, '25%': 4.0, '50%': 7.0, '75%': 12.0, 'max': 475.0}
non-ASCII rows (input): 446 | zero-width/control chars: 0
distinct input texts among rows with input: 18362 of 19157


### A.2 Preprocessing
| Step | Why |
|---|---|
| Keep only rows with a non-empty `input` (subset rule) | only these have a "data" part into which an attack can be injected, exactly as in StruQ; rows without input stay in the raw file |
| Unicode NFC, strip control / zero-width characters, trim trailing spaces, collapse 3+ blank lines | consistent text; **internal spaces are kept** because inputs include code and tables |
| Drop empty outputs and exact duplicates of (instruction, input) | duplicates inflate results and can leak across splits |
| Flag inputs that already contain instruction-like phrasing (`instruction_like`, regex heuristic) | these are natural benign hard cases for the false-positive rate; the flag is a screening aid, **not** a ground-truth label |
| Language tag; input-length buckets from the train split only | planned language and length analysis |
| Group key = normalised input text | the same input text appears with several instructions; all of them must stay in one split |

In [3]:
INSTR = re.compile(r"\b(?:ignore (?:all |any )?(?:previous|above|prior)|disregard|from now on|new instructions?|you (?:must|should|need to)|please (?:do|write|send|reply|answer|print)|instead,? (?:please )?)\b", re.I)
df = raw[has_in].copy()
n0 = len(df)
for c in ["instruction", "input", "output"]: df[c] = df[c].map(clean_text)
df = df[(df.input != "") & (df.output != "")]; n1 = len(df)
df = df.drop_duplicates(["instruction", "input"]).reset_index(drop=True); n2 = len(df)
print(f"rows: with-input {n0} -> non-empty {n1} -> after dedup {n2}")
df["input_group"] = df.input.map(lambda s: md5(re.sub(r"\s+", " ", s.lower()).strip())[:12])
df["id"] = ["alp-" + md5(i + "||" + x)[:10] for i, x in zip(df.instruction, df.input)]
assert df.id.is_unique
df["instruction_like"] = df.input.str.contains(INSTR)
df["language"] = df.input.map(lang_tag)
df["words"] = df.input.str.split().str.len()
print(df.language.value_counts().to_dict(), "| instruction-like inputs:", int(df.instruction_like.sum()))
print("rows sharing an input text with another row:", int(df.duplicated("input_group", keep=False).sum()), "| distinct input groups:", df.input_group.nunique())

rows: with-input 19157 -> non-empty 19157 -> after dedup 19154


{'en': 19154} | instruction-like inputs: 62
rows sharing an input text with another row: 1315 | distinct input groups: 18274


### A.3 Split (before any balancing)
Random 70/15/15 **at input-group level**, seed 42, so one input text never appears in two splits. Train + validation = development; test untouched until the final comparison.

In [4]:
train, val, test = group_split(df, "input_group")
q = add_buckets([train, val, test], "words")
allp = pd.concat([train, val, test])
print("tertile thresholds (input words, train):", q.round(1).to_dict())
print(allp.groupby("split").agg(rows=("id", "size"), groups=("input_group", "nunique"), instruction_like=("instruction_like", "sum")))

tertile thresholds (input words, train): {'q1': 5.0, 'q2': 10.0}
        rows  groups  instruction_like
split                                 
test    2888    2742                10
train  13401   12791                38
val     2865    2741                14


### A.4 Before / after

In [5]:
print(pd.DataFrame({"rows": [len(raw), len(df)], "rows_with_input": [int(has_in.sum()), len(df)],
    "mean_input_words": [round(raw[has_in].input.str.split().str.len().mean(), 1), round(df.words.mean(), 1)]}, index=["raw file", "processed"]))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(raw[has_in].input.str.len().clip(upper=1500), bins=40, alpha=.55, label="before"); ax[0].hist(df.input.str.len().clip(upper=1500), bins=40, alpha=.55, label="after")
ax[0].set_title("Input length (characters, clipped at 1500)"); ax[0].legend()
allp.groupby(["split", "length_bucket"]).size().unstack().plot(kind="bar", stacked=True, ax=ax[1], title="Length buckets per split", rot=0)
plt.tight_layout(); plt.savefig(FIG / "struq_alpaca_before_after.png", dpi=150)
print("\nSAMPLE:", allp.iloc[0].instruction, "|", allp.iloc[0].input[:150])

            rows  rows_with_input  mean_input_words
raw file   51760            19157              11.3
processed  19154            19154              11.3



SAMPLE: Evaluate this sentence for spelling and grammar mistakes | He finnished his meal and left the resturant


## Part B – StruQ attack templates (from `config.py`)

In [6]:
tree = ast.parse((REPO / "config.py").read_text(encoding="utf-8"))
consts = {}
for node in tree.body:
    if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name) and node.targets[0].id in ("IGNORE_ATTACK_SENTENCES", "TEST_INJECTED_WORDS"):
        consts[node.targets[0].id] = ast.literal_eval(node.value)
rows = []
role = {"train": "development", "test": "final_test", "hackaprompt": "task_specific_translation"}
for fam, items in consts["IGNORE_ATTACK_SENTENCES"].items():
    for i, t in enumerate(items):
        rows.append(dict(family="ignore" if fam != "hackaprompt" else "hackaprompt", struq_key=fam, role=role[fam], template=clean_text(t),
                         has_placeholder="{injected_prompt}" in t))
att = pd.DataFrame(rows); n_att_raw = len(att)
print("duplicate templates dropped:", int(att.template.duplicated().sum()))
att = att.drop_duplicates("template").reset_index(drop=True); att["words"] = att.template.str.split().str.len(); att["language"] = att.template.map(lang_tag)
att["attack_id"] = ["struq-att-%02d" % i for i in range(len(att))]
print("templates:", att.groupby(["struq_key", "role"]).size().to_dict())
print("injected test words used by StruQ:", consts["TEST_INJECTED_WORDS"], "-> test prompt: Print exactly 'Hacked!'")
print("templates:", n_att_raw, "->", len(att), "| without {injected_prompt} placeholder:", int((~att.has_placeholder).sum()))

duplicate templates dropped: 1
templates: {('hackaprompt', 'task_specific_translation'): 19, ('test', 'final_test'): 11, ('train', 'development'): 10}
injected test words used by StruQ: ['Hacked', 'Hello World', 'Serendipity'] -> test prompt: Print exactly 'Hacked!'
templates: 41 -> 40 | without {injected_prompt} placeholder: 1


## Save

In [7]:
cols = ["id", "input_group", "split", "instruction", "input", "output", "instruction_like", "language", "words", "length_bucket"]
for name, x in [("train", train), ("val", val), ("test", test)]:
    x[cols].to_csv(OUT / f"alpaca_cleaned_input_{name}.csv", index=False, encoding="utf-8")
att[["attack_id", "family", "struq_key", "role", "template", "has_placeholder", "words", "language"]].to_csv(OUT / "struq_attack_templates.csv", index=False, encoding="utf-8")
report = dict(dataset="StruQ / Alpaca-cleaned", code_source="https://github.com/Sizhe-Chen/StruQ", code_commit=commit, code_licence="CC BY-NC 4.0 (non-commercial)",
    data_source=ALPACA_URL, seed=SEED, raw_rows=int(len(raw)), rows_with_input=int(has_in.sum()), clean_rows=int(len(df)),
    split_sizes={s: int(len(x)) for s, x in [("train", train), ("val", val), ("test", test)]}, instruction_like_inputs=int(df.instruction_like.sum()),
    attack_templates_raw=int(n_att_raw), attack_templates=int(len(att)), language_counts=df.language.value_counts().to_dict(),
    not_included="AlpacaEval davinci_003_outputs.json (Hugging Face host, unreachable here)")
json.dump(report, open(OUT / "struq_preprocessing_report.json", "w"), indent=2); print(json.dumps(report, indent=2))

{
  "dataset": "StruQ / Alpaca-cleaned",
  "code_source": "https://github.com/Sizhe-Chen/StruQ",
  "code_commit": "e4372b9e3d45ecdfc7c0ad70662094efe6e3850b",
  "code_licence": "CC BY-NC 4.0 (non-commercial)",
  "data_source": "https://raw.githubusercontent.com/gururise/AlpacaDataCleaned/refs/heads/main/alpaca_data_cleaned.json",
  "seed": 42,
  "raw_rows": 51760,
  "rows_with_input": 19157,
  "clean_rows": 19154,
  "split_sizes": {
    "train": 13401,
    "val": 2865,
    "test": 2888
  },
  "instruction_like_inputs": 62,
  "attack_templates_raw": 41,
  "attack_templates": 40,
  "language_counts": {
    "en": 19154
  },
  "not_included": "AlpacaEval davinci_003_outputs.json (Hugging Face host, unreachable here)"
}
